# 基本クラスタ量をColabで見る（H₂を使わない解析）

**「ランタイム → すべてのセルを実行」**で、今回の18,300試料（L=32,64,128、各占有率100試料）から7種類の図を作ります。CPUで実行できます。

シミュレーションは再実行せず、保存済みの試料別 `samples.csv` を再集計します。`summary.csv` や `size_relation.csv` ではありません。

初回のみ、Driveに入力がなければPCの今回の結果フォルダにある **samples.csv** をアップロードしてください。以降はDriveの保存済み入力を使います。図・CSV・報告用の説明文をDriveに保存し、最後に共有用ZIPをダウンロードできます。


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/tetra-peg-network/cluster_basics')
ROOT.mkdir(parents=True, exist_ok=True)
# 別の場所に入力がある場合は、そのsamples.csvのパスに変更します。
INPUT = ROOT / 'input' / 'samples.csv'


## 1. 入力データの選択

既定の保存先に入力がなければ、以前の `h2_report/samples.csv` も確認します。どちらにもなければ、ファイル選択で `samples.csv` をアップロードします。入力が既にあれば選択操作はありません。


In [ ]:
from google.colab import files
import csv
import hashlib

if not INPUT.is_file():
    previous_input = ROOT.parent / 'h2_report' / 'samples.csv'
    if previous_input.is_file():
        INPUT = previous_input
    else:
        print('今回の100試料/条件の結果フォルダにある samples.csv を選択してください。')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('samples.csvを1ファイルだけ選択してください。')
        name, data = next(iter(uploaded.items()))
        if Path(name).name != 'samples.csv':
            raise ValueError('summary.csv等ではなく、samples.csvを選択してください。')
        # 不正なファイルを既定の保存先に残さないため、列・件数を先に確認。
        import io
        reader = csv.DictReader(io.StringIO(data.decode('utf-8-sig')))
        required = {'L', 'target_phi', 'actual_phi', 'N', 'N1', 'bonds', 'clusters',
                    'cluster_number_mean', 'cluster_weighted_all',
                    'cluster_weighted_excl_one', 'seed'}
        if not required.issubset(set(reader.fieldnames or [])):
            raise ValueError('試料別CSVの必須列がありません。今回のsamples.csvを確認してください。')
        rows = list(reader)
        from collections import Counter
        counts = Counter((int(r['L']), round(float(r['target_phi']), 8)) for r in rows)
        expected = {(L, round(k/100, 8)) for L in (32,64,128) for k in range(10,71)}
        if set(counts) != expected or any(n != 100 for n in counts.values()):
            raise ValueError('今回の183条件・各100試料と一致しません。入力ファイルを確認してください。')
        INPUT.parent.mkdir(parents=True, exist_ok=True)
        INPUT.write_bytes(data)
        del uploaded, data, rows
print('入力:', INPUT)
print('入力SHA256:', hashlib.sha256(INPUT.read_bytes()).hexdigest())


## 2. PC版と同じコード・設定を取得

コードと設定を同じGitHubコミットに固定し、SHA256も確認します。図は平均±標準誤差（SE）です。


In [ ]:
import urllib.request
import subprocess
import sys

REPOSITORY = 'takakei510/tetra-peg-network'
REVISION = '264b9d5bf24b765dc031b2ccc47622241e0ab300'
WORK = Path('/content/cluster_basics_code') / REVISION
WORK.mkdir(parents=True, exist_ok=True)
SOURCES = {
    'script': ('scripts/analysis/plot_cluster_basics.py', '2468d7c2be665437e258cea174c5f435017c21e6d569248da425dda783dd08a4'),
    'config': ('configs/analysis/cluster-basics.json', '5687a6a0f8a5a8690b8085c299be04f7a06b99ed3e0ecd1c55468f3540d31d94'),
}
local_files = {}
for key, (remote, expected_sha) in SOURCES.items():
    url = f'https://raw.githubusercontent.com/{REPOSITORY}/{REVISION}/{remote}'
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    if hashlib.sha256(data).hexdigest() != expected_sha:
        raise ValueError(f'{remote} のSHA256が一致しません。')
    dest = WORK / Path(remote).name
    dest.write_bytes(data)
    local_files[key] = dest
    print('取得:', dest.name)

FONT = Path('/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc')
if not FONT.is_file():
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'], check=True)
if not FONT.is_file():
    raise FileNotFoundError('日本語フォントの準備に失敗しました。')


## 3. 作図・保存

毎回新しい日時のフォルダに保存します。入力の整合性・重複・各条件100試料を検査してから集計します。未定義の有限クラスタ平均は0に置き換えず、有効試料数をCSVに記録します。


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

stamp = datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT = ROOT / f'figures_{stamp}'
command = [sys.executable, str(local_files['script']),
           '--config', str(local_files['config']), '--results', str(INPUT),
           '--output', str(OUTPUT), '--font', str(FONT)]
subprocess.run(command, check=True)
print('コード固定コミット:', REVISION)
print('Drive保存先:', OUTPUT)


## 4. 図と目的の確認

- **最大クラスタ**：全分子のうち、どれだけが一つの集団に属するか。絶対数と割合を分けます。
- **平均サイズ**：クラスタ等重みの平均と、分子を起点とする重み付き平均を区別します。最大クラスタを除いた平均も比較します。
- **結合率・クラスタ数**：末端結合と大きな集団形成の対応、集団の個数と密度を見ます。
- **ピーク・試料数**：有限サイズによるピーク位置の違いと、今回の100試料の部分集合での平均・SEの変化を確認します。

図ごとの定義・報告での説明例・解釈の限界は、下に表示する `figure_guide.md` にあります。Sfinの最大点は有限サイズ・測定点中の最大であり、無限系の臨界点と断定しません。試料数の図は前回の5試料そのものとの比較ではありません。


In [ ]:
from IPython.display import display, Image, Markdown

for image_path in sorted(OUTPUT.glob('*.png')):
    display(Markdown(f'### {image_path.stem}'))
    display(Image(filename=str(image_path)))
display(Markdown((OUTPUT / 'figure_guide.md').read_text(encoding='utf-8')))


## 5. ChatGPTへ結果を共有

下のボタンで `cluster_basics_share.zip` をダウンロードし、チャットに添付してください。7種類のPNG/SVG、集計CSV、図の説明、入力とコードのSHA256を含む実行記録が入っています。同じZIPはDriveの今回の出力フォルダにもあります。


In [ ]:
import ipywidgets as widgets

archive = OUTPUT / 'cluster_basics_share.zip'
button = widgets.Button(description='共有用ZIPをダウンロード', layout=widgets.Layout(width='260px'))
button.on_click(lambda _: files.download(str(archive)))
display(button)
print('ZIP:', archive)
